# simpleNMR → JASON assignment write-back — version 3

Reads `nodes_now` from the simpleNMR JSON and writes the assignments into the JASON `.jjh5`
with `Molecule.set_assignments()` (beautifuljason ≥ 1.3.0b1).

* **Atoms** – simpleNMR node `id` *i* is JASON atom *i*. Elements and bonds are checked; the notebook stops if they differ.
* **13C** – the node's `ppm` is looked up exactly in the 1D 13C spectra → `(shift, peak)`.
* **1H** – each value in `H1_ppm` is looked up exactly on the 1H axis of the 1H/13C 2D spectra; if more than
  one peak matches, the one whose 13C coordinate is closest to the node's carbon wins (the HSQC peak) → `(shift, peak, h_dim)`.
* **CH₂** – two predicted shifts marked `(dn)`/`(up)` are paired with the two proton values in ascending ppm;
  a single value is used for both (as in the original notebook).

HMBC/COSY peaks are not assigned: JASON draws those correlations itself once the atoms are assigned.
`DRY_RUN = True` by default; a timestamped backup is made before writing.

In [ ]:
from pathlib import Path

SIMPLENMR_JSON = Path('input_py.json')
JJH5_FILE      = Path('input.jjh5')
MOLECULE_INDEX = 0

DRY_RUN     = True
MAKE_BACKUP = True
PPM_EPS     = 1e-4     # floating-point allowance for "same value" (ppm), not a chemical tolerance

In [ ]:
import json
import math
import shutil
from datetime import datetime

import pandas as pd
import beautifuljason as bjason

H1  = bjason.Molecule.Atom.NuclType.H1
C13 = bjason.Molecule.Atom.NuclType.C13
ELEMENT_H = int(bjason.Molecule.Atom.Type.H)

print('beautifuljason', getattr(bjason, '__version__', '?'),
      '| set_assignments available:', hasattr(bjason.Molecule, 'set_assignments'))

## 1 · simpleNMR nodes

In [ ]:
def ppm_or_none(v, limit=1000):
    """float, or None for missing values and simpleNMR's 10000 placeholder."""
    try:
        v = float(v)
    except (TypeError, ValueError):
        return None
    return v if math.isfinite(v) and abs(v) < limit else None


snmr = json.loads(SIMPLENMR_JSON.read_text())

nodes = []                                   # carbons only, keyed by 0-based id
for n in snmr['nodes_now']:
    if n.get('symbol') != 'C':
        continue
    h_ppms = sorted(v for v in (ppm_or_none(x, 100) for x in n.get('H1_ppm') or []) if v is not None)
    nodes.append({'id': int(n['id']), 'label': f"C{int(n['id']) + 1}", 'nH': int(n.get('numProtons') or 0),
                  'c_ppm': ppm_or_none(n.get('ppm')), 'h_ppms': h_ppms})
nodes.sort(key=lambda n: n['id'])

snmr_elements = {int(n['id']): int(n['atom_number']) for n in snmr['molgraph']['nodes']}
snmr_bonds = {frozenset((int(l['source']), int(l['target']))) for l in snmr['molgraph']['links']}

print('SMILES:', snmr.get('smilesString'))
pd.DataFrame(nodes)

## 2 · Helpers

In [ ]:
def check_atoms(molecule):
    elements = {i: int(a.type) for i, a in enumerate(molecule.atoms) if int(a.type) != ELEMENT_H}
    bonds = {frozenset((i, int(j))) for i in elements
             for j in molecule.atoms[i].bonded_atom_numbers if int(j) in elements}
    if elements != snmr_elements or bonds != snmr_bonds:
        raise ValueError(
            'simpleNMR and JASON molecules do not match atom-for-atom\n'
            f'  element differences: {sorted(set(elements.items()) ^ set(snmr_elements.items()))}\n'
            f'  bonds only in JASON: {sorted(tuple(sorted(b)) for b in bonds - snmr_bonds)}\n'
            f'  bonds only in simpleNMR: {sorted(tuple(sorted(b)) for b in snmr_bonds - bonds)}')


def all_peaks(document, ndim, nuclei):
    """[(nmr_data index, peak)] for every spectrum with this dimensionality and nuclei."""
    out = []
    for i, s in enumerate(document.nmr_data):
        if s.ndim == ndim and set(s.spec_info.nuclides) == set(nuclei):
            try:
                out += [(i, p) for p in s.peaks]
            except KeyError:
                pass
    return out


def predicted_shifts(molecule, nucleus):
    """{atom id: [(shift, predicted ppm, label)]} in ascending ppm; 1H shifts are keyed by their carbon."""
    specs = [s for s in molecule.spectra if s.nucleus == nucleus]
    out = {}
    if not specs:
        return out
    prefix = 'H' if nucleus == H1 else 'C'
    for shift in max(specs, key=lambda s: len(s.shifts)).shifts:
        if shift.ignored_auto or shift.ignored_user:
            continue
        value, _ = shift.get_value_error_pair(bjason.Molecule.CalcMethod.Best)
        value = None if value is None else float(value)
        mark = f'({shift.mark})' if shift.mark else ''
        for n in map(int, shift.nums):
            out.setdefault(n, []).append((shift, value, f'{prefix}{n + 1}{mark}'))
    for lst in out.values():
        lst.sort(key=lambda t: (t[1] is None, t[1]))
    return out

## 3 · Build the `(shift, peak[, dim])` tuples

In [ ]:
def build_assignments(document):
    molecule = document.mol_data[MOLECULE_INDEX]
    check_atoms(molecule)

    c13_peaks = all_peaks(document, 1, ('13C',))
    hc_peaks = []                                  # (index, peak, h_dim, c_dim) for HSQC/HMBC-type spectra
    for i, p in all_peaks(document, 2, ('1H', '13C')):
        nuc = tuple(document.nmr_data[i].spec_info.nuclides)
        hc_peaks.append((i, p, nuc.index('1H'), nuc.index('13C')))

    pred_c, pred_h = predicted_shifts(molecule, C13), predicted_shifts(molecule, H1)
    tuples, rows = [], []

    def record(node, nucleus, shift_label, predicted, snmr_ppm, source, status):
        rows.append({'atom': node['label'], 'nucleus': nucleus, 'shift': shift_label, 'predicted': predicted,
                     'simpleNMR': snmr_ppm, 'source': source, 'status': status})

    for node in nodes:
        # ---- 13C: exact match in a 1D 13C spectrum
        c = node['c_ppm']
        hits = [(i, p) for i, p in c13_peaks if c is not None and abs(float(p.pos[0]) - c) <= PPM_EPS]
        c_shifts = pred_c.get(node['id'], [])
        if c is None:
            record(node, '13C', '', None, None, '', 'no 13C value in simpleNMR')
        elif not hits:
            record(node, '13C', '', None, c, '', 'NO EXACT 13C PEAK')
        elif not c_shifts:
            record(node, '13C', '', None, c, f'nmr_data[{hits[0][0]}]', 'no predicted 13C shift')
        else:
            i, peak = hits[0]
            extra = f' ({len(hits)} matching peaks, first used)' if len(hits) > 1 else ''
            for shift, value, label in c_shifts:
                tuples.append((shift, peak))
                record(node, '13C', label, value, c, f'nmr_data[{i}]', 'assigned' + extra)

        # ---- 1H: exact match on the 1H axis; closest 13C coordinate picks the HSQC peak
        signals = []
        for h in node['h_ppms']:
            hits = [t for t in hc_peaks if abs(float(t[1].pos[t[2]]) - h) <= PPM_EPS]
            if not hits:
                record(node, '1H', '', None, h, '', 'NO EXACT 1H PEAK')
                continue
            if c is not None:
                hits.sort(key=lambda t: abs(float(t[1].pos[t[3]]) - c))
            signals.append((h, *hits[0]))
        h_shifts = pred_h.get(node['id'], [])
        if signals and not h_shifts:
            record(node, '1H', '', None, signals[0][0], '', 'no predicted 1H shift')
            continue
        if len(signals) == 1 and len(h_shifts) > 1:          # CH2 with one value: use it for both shifts
            signals = signals * len(h_shifts)
        for (shift, value, label), (h, i, peak, hd, cd) in zip(h_shifts, signals):
            tuples.append((shift, peak, hd))
            record(node, '1H', label, value, h, f'nmr_data[{i}]', 'assigned')

    return molecule, tuples, pd.DataFrame(rows)

## 4 · Dry run (read-only)

In [ ]:
with bjason.Document(str(JJH5_FILE), mode='r') as doc:
    _, tuples, report = build_assignments(doc)

pd.set_option('display.float_format', '{:.4f}'.format)
print(f"{len(tuples)} assignment tuples; {(report['status'] != 'assigned').sum()} rows need attention")
report

## 5 · Write
Set `DRY_RUN = False` in the configuration cell, re-run it, then run this cell.

In [ ]:
if DRY_RUN:
    print('DRY_RUN is True – nothing written.')
else:
    if MAKE_BACKUP:
        backup = JJH5_FILE.with_name(f"{JJH5_FILE.stem}.{datetime.now():%Y%m%d-%H%M%S}.bak{JJH5_FILE.suffix}")
        shutil.copy2(JJH5_FILE, backup)
        print('Backup:', backup)
    with bjason.Document(str(JJH5_FILE), mode='r+') as doc:      # rebuild: peaks belong to the open file
        molecule, tuples, report = build_assignments(doc)
        molecule.set_assignments(tuples)
    print(f'Wrote {len(tuples)} assignments to {JJH5_FILE}')